# Per-Category Evaluation

Loads a trained checkpoint, runs the COCO evaluator, and shows per-category metrics.

Produces:
- Horizontal bar chart of AP50 per category (sorted)
- Top-5 and bottom-5 categories by AP50
- Distance MAE / absrel (if the model was trained with distance)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
config_path      = "configs/experiments/yolo/yolov8_poly_dist.yaml"
checkpoint_path  = "/tmp/yolo_run/ckpt-5000"  # e.g. path prefix to a checkpoint

import os, sys
# Ensure repo root is on the path when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Config:     {config_path}")
print(f"Checkpoint: {checkpoint_path}")

## Build Model and Run Evaluation

In [ ]:
import tensorflow as tf
tf.config.run_functions_eagerly(False)

from configs.yaml_loader import load_config
from models.yolo_v8 import build_yolov8
from train.task import YoloV8Task
from eval.coco_metrics import COCOEvaluator
from eval.distance_metrics import DistanceEvaluator

config = load_config(config_path)
task   = YoloV8Task(config)

# Build and restore model
model = build_yolov8(config.task.model)
model.deploy = True
model.build_and_init(config.task.model.input_size)

ckpt = tf.train.Checkpoint(model=model)
ckpt.restore(checkpoint_path).expect_partial()
print("Checkpoint restored.")

# Build validation dataset
import dataclasses
val_cfg = dataclasses.replace(config.task.validation_data, is_training=False)
val_ds  = task.build_inputs(val_cfg)

img_size = tuple(config.task.model.input_size[:2])
coco_ev  = COCOEvaluator(num_classes=config.task.num_classes, image_size=img_size)
dist_ev  = DistanceEvaluator() if config.task.with_distance else None

print("Running evaluation...")
for step, (images, labels) in enumerate(val_ds):
    preds = model(images, training=False)
    coco_ev.update(preds, labels)
    if dist_ev:
        n_gt  = labels["n_gt"].numpy()
        gt_ld = labels["log_distance"].numpy()
        pd_d  = preds["distance"].numpy()
        nd    = preds["num_detections"].numpy()
        for i in range(len(n_gt)):
            if n_gt[i] > 0 and nd[i] > 0:
                dist_ev.update(pd_d[i, :nd[i]], gt_ld[i, :n_gt[i]])
    if step % 50 == 0:
        print(f"  step {step}")

overall = coco_ev.evaluate()
if dist_ev:
    overall.update(dist_ev.evaluate())

print("\n=== Overall Metrics ===")
for k, v in sorted(overall.items()):
    print(f"  {k:25s}: {v:.4f}")

## Per-Category Metrics

In [ ]:
per_cat = coco_ev.per_category_full_metrics()

cat_ids  = sorted(per_cat.keys())
ap50s    = [per_cat[c]["ap50"]  for c in cat_ids]
aps      = [per_cat[c]["ap"]    for c in cat_ids]
ap75s    = [per_cat[c]["ap75"]  for c in cat_ids]
ar100s   = [per_cat[c]["ar100"] for c in cat_ids]

print(f"{'Cat':>4}  {'AP50':>6}  {'AP':>6}  {'AP75':>6}  {'AR100':>6}")
for cat_id, ap50, ap, ap75, ar100 in zip(cat_ids, ap50s, aps, ap75s, ar100s):
    print(f"{cat_id:>4}  {ap50:>6.4f}  {ap:>6.4f}  {ap75:>6.4f}  {ar100:>6.4f}")

## AP50 Bar Chart (Sorted)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Sort by AP50 descending
order   = np.argsort(ap50s)[::-1]
s_cats  = [cat_ids[i]  for i in order]
s_ap50s = [ap50s[i]    for i in order]

fig_h = max(6, len(s_cats) * 0.3)
fig, ax = plt.subplots(figsize=(10, fig_h))
y_pos = np.arange(len(s_cats))
ax.barh(y_pos, s_ap50s, align="center", color="steelblue", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"cat {c}" for c in s_cats], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("AP50")
ax.set_title(f"AP50 per Category  (mean={np.mean(s_ap50s):.4f})")
ax.axvline(np.mean(s_ap50s), color="red", linestyle="--", linewidth=1, label="mean")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()

plot_path = "ap50_per_category.png"
fig.savefig(plot_path, dpi=120)
plt.close(fig)
print(f"Plot saved: {plot_path}")

## Top-5 and Bottom-5 Categories

In [ ]:
k = 5
print(f"Top-{k} categories by AP50:")
for rank, i in enumerate(order[:k], 1):
    print(f"  #{rank}  cat {cat_ids[i]:>3}  AP50={ap50s[i]:.4f}  AP={aps[i]:.4f}  AR100={ar100s[i]:.4f}")

print(f"\nBottom-{k} categories by AP50:")
for rank, i in enumerate(order[-k:][::-1], 1):
    print(f"  #{rank}  cat {cat_ids[i]:>3}  AP50={ap50s[i]:.4f}  AP={aps[i]:.4f}  AR100={ar100s[i]:.4f}")

if dist_ev:
    dist_metrics = dist_ev.evaluate()
    print("\n=== Distance Metrics ===")
    mae    = dist_metrics.get("dist_mae", None)
    rmse   = dist_metrics.get("dist_rmse", None)
    if mae is not None:
        print(f"  MAE  : {mae:.4f} m")
    if rmse is not None:
        # absrel uses overall mean GT distance; approximate from RMSE here
        print(f"  RMSE : {rmse:.4f} m")